# 🔥 Setup & Migrasi Presensi → Firebase (sekali jalan)
Memindahkan **Google Sheet v11.1** ke **Firebase Realtime Database** + membuat akun admin, alat (ESP32), dan **akun orang tua (NIS + PIN)**.
Aman dijalankan ulang: akun/PIN yang sudah ada **tidak** diubah (kecuali password alat — selalu dibuat baru).

### Persiapan di Firebase Console (project `presensi-ma`)
1. **Build → Realtime Database → Create database** → lokasi **Singapore (asia-southeast1)** → mode **Locked**.
   Salin URL database (contoh `https://presensi-ma-7c521-default-rtdb.asia-southeast1.firebasedatabase.app`).
2. **Realtime Database → tab Rules** → hapus isinya, tempel isi file **`database.rules.json`** → **Publish**.
3. **Build → Authentication → Get started → Sign-in method → Email/Password → Enable** (yang pertama saja, *Email link* tidak perlu).
4. **⚙️ Project settings → Service accounts → Generate new private key** → file `.json` terunduh.
   ⚠️ File ini = kunci penuh project. Jangan dibagikan/di-upload ke GitHub; hapus dari Colab setelah selesai.
5. **Google Cloud Console** (project yang sama) → *APIs & Services → Library* → aktifkan **Google Sheets API** & **Google Drive API**.
6. Buka **spreadsheet presensi** → **Share** ke email `client_email` di file JSON tadi sebagai **Viewer**.

### Jalankan
Runtime → **Run all**, lalu ikuti input di sel 2. Hasil akhir yang diunduh:
- `PIN_ortu_*.csv` — daftar **NIS + PIN** untuk dibagikan TU ke orang tua (rahasia)
- `kredensial_esp32.txt` — email & password alat untuk firmware v12 (rahasia)

In [ ]:
# 1) Install
!pip -q install firebase-admin gspread
print("Selesai")

In [ ]:
# 2) Konfigurasi
import os, json, getpass
from google.colab import files

DB_URL = "https://ISI-URL-DATABASE.asia-southeast1.firebasedatabase.app"   # dari langkah 1
SPREADSHEET_ID = "ISI_ID_SPREADSHEET"                                     # docs.google.com/spreadsheets/d/<ID>/edit
ADMIN_EMAIL = "admin@manurulhuda.sch.id"                                   # email login admin web (boleh fiktif)

if not os.path.exists("/content/sa.json"):
    print("Upload file JSON service account Firebase:")
    up = files.upload(); os.rename(list(up.keys())[0], "/content/sa.json")
sa_email = json.load(open("/content/sa.json"))["client_email"]
print("Spreadsheet harus di-SHARE (Viewer) ke:", sa_email)
ADMIN_PASSWORD = getpass.getpass("Password admin web (min. 8 karakter): ")
assert len(ADMIN_PASSWORD) >= 8, "Password admin terlalu pendek"
assert DB_URL.startswith("https://") and "ISI-URL" not in DB_URL, "Isi DB_URL dulu"
assert "ISI_ID" not in SPREADSHEET_ID, "Isi SPREADSHEET_ID dulu"

In [ ]:
%%writefile /content/migrasi.py
"""
Setup & migrasi Presensi MA Nurul Huda: Google Sheet v11.1 -> Firebase RTDB.
Dijalankan SEKALI dari Colab (lihat notebook). Idempoten: aman diulang.

Yang dilakukan:
  1. Akun admin & device + /roles
  2. Data_Induk  -> /config/students/{UID}, /profiles/{NIS}
  3. Akun ortu (email {NIS}.{ver}@ORTU_DOMAIN, password = PIN 6 digit acak) -> /parents/{NIS}
     + CSV daftar NIS & PIN untuk dibagikan TU
  4. Data_Jadwal -> /config/schedule
  5. Data_UtamaSiswa -> /attendance/{NIS}/{tgl} & /log/{tgl}/{UID}_{M|P}
     Data_Izin (opsional) -> /attendance/{NIS}/{tgl}/ket
  6. Naikkan /config/version  (ESP32 otomatis unduh ulang)
"""
import csv
import re
import secrets
import time
from datetime import datetime

import firebase_admin
from firebase_admin import auth, credentials, db

ORTU_DOMAIN = "ortu.presensi-ma.local"     # domain fiktif; HARUS sama dengan konstanta di web
DEVICE_EMAIL = "esp32-01@device.presensi-ma.local"
BATCH = 400                                  # path per update() agar request tidak terlalu besar
RE_UID = re.compile(r"^[0-9A-F]{8,20}$")
RE_JAM = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")


def email_ortu(nis: str, ver: int) -> str:
    return f"{nis}.{ver}@{ORTU_DOMAIN}"


def buat_pin() -> str:
    return f"{secrets.randbelow(1_000_000):06d}"   # CSPRNG, bukan random.random


def init(sa_file: str, db_url: str):
    if not firebase_admin._apps:
        firebase_admin.initialize_app(credentials.Certificate(sa_file), {"databaseURL": db_url})


def _update_batch(paths: dict, label: str):
    items = list(paths.items())
    for i in range(0, len(items), BATCH):
        db.reference("/").update(dict(items[i:i + BATCH]))
    print(f"  {label}: {len(items)} path ditulis")


def _user_by_email(email):
    try:
        return auth.get_user_by_email(email)
    except auth.UserNotFoundError:
        return None


# ------------------------------------------------------------------ 1. akun inti
def setup_akun_inti(admin_email: str, admin_password: str) -> dict:
    """Admin pakai password pilihan Anda; device dapat password acak 24 karakter."""
    hasil = {}
    u = _user_by_email(admin_email)
    if u is None:
        u = auth.create_user(email=admin_email, password=admin_password)
        print("  akun admin dibuat:", admin_email)
    else:
        print("  akun admin sudah ada:", admin_email)
    hasil["admin_uid"] = u.uid

    d = _user_by_email(DEVICE_EMAIL)
    dev_pw = secrets.token_urlsafe(18)
    if d is None:
        d = auth.create_user(email=DEVICE_EMAIL, password=dev_pw)
    else:
        auth.update_user(d.uid, password=dev_pw)   # rotasi: password lama tidak bisa diambil kembali
    hasil.update(device_uid=d.uid, device_email=DEVICE_EMAIL, device_password=dev_pw)

    db.reference("roles").update({u.uid: "admin", d.uid: "device"})
    return hasil


# ------------------------------------------------------------------ 2+3. siswa & ortu
def migrasi_siswa(induk_rows: list, csv_path: str) -> dict:
    """induk_rows: baris Data_Induk TANPA header: [UID, Nama, Kelas, NIS]."""
    siswa, masalah, nis_ke_uid = {}, [], {}
    for i, r in enumerate(induk_rows, start=2):
        r = [str(x).strip() for x in (list(r) + [""] * 4)[:4]]
        uid, nama, kelas, nis = r[0].upper(), r[1][:60], r[2][:15], r[3]
        if not uid:
            continue
        if not RE_UID.match(uid):
            masalah.append(f"baris {i}: UID tidak valid '{uid}'")
            continue
        if not re.fullmatch(r"\d{1,20}", nis):
            masalah.append(f"baris {i}: NIS kosong/tidak valid '{nis}' ({nama})")
            continue
        if uid in siswa:
            masalah.append(f"baris {i}: UID ganda {uid}")
            continue
        if nis in nis_ke_uid:
            masalah.append(f"baris {i}: NIS ganda {nis}")
            continue
        siswa[uid] = {"nis": nis, "nama": nama, "kelas": kelas}
        nis_ke_uid[nis] = uid

    paths = {}
    for uid, s in siswa.items():
        paths[f"config/students/{uid}"] = s
        paths[f"profiles/{s['nis']}"] = {"nama": s["nama"], "kelas": s["kelas"]}
    _update_batch(paths, "siswa & profil")

    # Akun ortu: hanya dibuat jika belum ada -> PIN lama tidak berubah saat skrip diulang
    ada = db.reference("parents").get() or {}
    baru, parents = [], {}
    for nis, uid in nis_ke_uid.items():
        if nis in ada and ada[nis].get("uid"):
            continue
        pin = buat_pin()
        email = email_ortu(nis, 1)
        u = _user_by_email(email)
        if u is None:
            u = auth.create_user(email=email, password=pin)
        else:
            auth.update_user(u.uid, password=pin)
        parents[f"parents/{nis}"] = {"uid": u.uid, "ver": 1}
        baru.append((nis, siswa[uid]["nama"], siswa[uid]["kelas"], pin))
    if parents:
        _update_batch(parents, "akun ortu baru")

    if baru:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(["NIS", "Nama", "Kelas", "PIN"])
            w.writerows(sorted(baru, key=lambda x: (x[2], x[1])))
    print(f"  {len(siswa)} siswa valid, {len(baru)} akun ortu baru, {len(masalah)} baris dilewati")
    return {"nis_ke_uid": nis_ke_uid, "masalah": masalah, "ortu_baru": len(baru)}


# ------------------------------------------------------------------ 4. jadwal
def migrasi_jadwal(jadwal_rows: list) -> int:
    """Data_Jadwal v11.1 tanpa header: [Kelas, Hari, Jenis, Jam_Mulai, Jam_Selesai]."""
    out, n = {}, 0
    for r in jadwal_rows:
        r = [str(x).strip() for x in (list(r) + [""] * 5)[:5]]
        if r[2] not in ("Masuk", "Terlambat", "Pulang") or not (RE_JAM.match(r[3]) and RE_JAM.match(r[4])):
            continue
        out[f"s{n:03d}"] = {"kelas": r[0], "hari": r[1], "jenis": r[2], "mulai": r[3][:5], "selesai": r[4][:5]}
        n += 1
    db.reference("config/schedule").set(out)
    print(f"  jadwal: {n} baris")
    return n


# ------------------------------------------------------------------ 5. riwayat
def _tgl(s: str):
    s = str(s).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%m/%d/%Y"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return None


def _jam(s: str):
    s = str(s).strip()
    if not RE_JAM.match(s):
        return None
    p = s.split(":")
    return f"{int(p[0]):02d}:{p[1]}:{p[2] if len(p) > 2 else '00'}"


def migrasi_riwayat(tap_rows: list, izin_rows: list, nis_ke_uid: dict) -> dict:
    """tap_rows: Data_UtamaSiswa tanpa header: [Tanggal, Jam, UID, Nama, Jabatan, Kelas, Status, NIS].
    Ambil tap Masuk & Pulang PERTAMA per siswa per tanggal (sama dgn aturan alat)."""
    uid_ke_nis = {u: n for n, u in nis_ke_uid.items()}
    masuk, pulang, lewat = {}, {}, 0
    for r in tap_rows:
        r = [str(x).strip() for x in (list(r) + [""] * 8)[:8]]
        tgl, jam, uid, status = _tgl(r[0]), _jam(r[1]), r[2].upper(), r[6]
        nis = r[7] or uid_ke_nis.get(uid, "")
        if not (tgl and jam and RE_UID.match(uid) and nis.isdigit()) or r[4] == "Guru":
            lewat += 1
            continue
        key = (nis, tgl)
        info = {"uid": uid, "jam": jam, "nama": r[3][:60], "kelas": r[5][:15], "status": status}
        if status in ("Hadir", "Terlambat"):
            if key not in masuk or jam < masuk[key]["jam"]:
                masuk[key] = info
        elif status == "Pulang":
            if key not in pulang or jam < pulang[key]["jam"]:
                pulang[key] = info
        else:
            lewat += 1

    paths = {}
    for (nis, tgl), m in masuk.items():
        paths[f"attendance/{nis}/{tgl}/masuk"] = m["jam"]
        paths[f"attendance/{nis}/{tgl}/status"] = m["status"]
        paths[f"log/{tgl}/{m['uid']}_M"] = {"jam": m["jam"], "nis": nis, "nama": m["nama"], "kelas": m["kelas"], "status": m["status"]}
    for (nis, tgl), p in pulang.items():
        paths[f"attendance/{nis}/{tgl}/pulang"] = p["jam"]
        paths[f"log/{tgl}/{p['uid']}_P"] = {"jam": p["jam"], "nis": nis, "nama": p["nama"], "kelas": p["kelas"], "status": "Pulang"}

    n_izin = 0
    for r in izin_rows or []:   # Data_Izin: [NIS, Tanggal, Jenis, Keterangan]
        r = [str(x).strip() for x in (list(r) + [""] * 4)[:4]]
        tgl = _tgl(r[1])
        jenis = "Sakit" if r[2].lower().startswith("sakit") else ("Izin" if r[2] else "")
        if r[0].isdigit() and tgl and jenis and (r[0], tgl) not in masuk:
            paths[f"attendance/{r[0]}/{tgl}/ket"] = jenis
            paths[f"izin/{tgl}/{r[0]}"] = jenis          # indeks harian utk dashboard admin
            if r[3]:
                paths[f"attendance/{r[0]}/{tgl}/catatan"] = r[3][:200]
            n_izin += 1

    _update_batch(paths, "riwayat presensi")
    print(f"  {len(masuk)} hari-masuk, {len(pulang)} hari-pulang, {n_izin} izin/sakit, {lewat} baris dilewati")
    return {"masuk": len(masuk), "pulang": len(pulang), "izin": n_izin, "lewat": lewat}


def naikkan_versi():
    v = int(time.time())
    db.reference("config/version").set(v)
    print("  config/version =", v)
    return v


In [ ]:
# 3) Baca Google Sheet
import gspread, sys
sys.path.insert(0, "/content")
ss = gspread.service_account(filename="/content/sa.json").open_by_key(SPREADSHEET_ID)

def baca(nama):
    try:
        return ss.worksheet(nama).get_all_values()[1:]      # tanpa header
    except gspread.WorksheetNotFound:
        print(f"  (sheet {nama} tidak ada, dilewati)"); return []

INDUK, JADWAL, TAP, IZIN = baca("Data_Induk"), baca("Data_Jadwal"), baca("Data_UtamaSiswa"), baca("Data_Izin")
print(f"Data_Induk {len(INDUK)} baris | Data_Jadwal {len(JADWAL)} | Data_UtamaSiswa {len(TAP)} | Data_Izin {len(IZIN)}")
# Cek format v11.1: Data_Induk kolom C = Kelas (bukan Jabatan), Data_Jadwal kolom A = Kelas
hdr_induk = ss.worksheet("Data_Induk").row_values(1)[:4]
print("Header Data_Induk:", hdr_induk)
assert hdr_induk[2].startswith("Kelas"), "Data_Induk belum format v11 (jalankan migrasiV11 di Apps Script dulu)"

In [ ]:
# 4) Migrasi ke Firebase
import migrasi as M
from datetime import datetime
M.init("/content/sa.json", DB_URL)

print("[1] Akun admin & alat");  AK = M.setup_akun_inti(ADMIN_EMAIL, ADMIN_PASSWORD)
csv_pin = f"/content/PIN_ortu_{datetime.now():%Y%m%d_%H%M}.csv"
print("[2] Siswa & akun orang tua"); S = M.migrasi_siswa(INDUK, csv_pin)
for m in S["masalah"]: print("    ⚠️", m)
print("[3] Jadwal");            M.migrasi_jadwal(JADWAL)
print("[4] Riwayat presensi");  M.migrasi_riwayat(TAP, IZIN, S["nis_ke_uid"])
print("[5] Versi data");        M.naikkan_versi()

with open("/content/kredensial_esp32.txt", "w") as f:
    f.write(f"DEVICE_EMAIL    = \"{AK['device_email']}\"\nDEVICE_PASSWORD = \"{AK['device_password']}\"\n")
print("\n✅ Selesai. Siswa yang dilewati (lihat ⚠️) perbaiki di Sheet lalu jalankan ulang sel ini.")

In [ ]:
# 5) Unduh hasil (RAHASIA — simpan aman, jangan kirim lewat grup)
import os
if os.path.exists(csv_pin): files.download(csv_pin)
else: print("Tidak ada akun ortu baru (semua sudah dibuat sebelumnya) — PIN lama tetap berlaku.")
files.download("/content/kredensial_esp32.txt")

In [ ]:
# 6) Bersihkan kunci dari Colab
for p in ["/content/sa.json", "/content/kredensial_esp32.txt"]:
    if os.path.exists(p): os.remove(p)
print("File rahasia dihapus dari Colab.")